# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [28]:
from langchain_community.document_loaders import PyPDFLoader
# Load the PDF document managing_oneself.pdf
file_path = "documents/managing_oneself.pdf"
# loader = PyPDFLoader("C:\\Users\\chauh\\OneDrive\\Documents\\manish\\ws\\UofT-DSI\\deploying-ai\\02_activities\\documents\\managing_oneself.pdf")
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
all_text = "\n".join([page.page_content for page in pages])

# Output:

print(f"Number of pages: {len(pages)}") 
# print(dict(pages[0]))
print("\n\n", pages[0].metadata)
print("\n all contents: ", all_text[:50])  # Print the first 500 characters of the combined text

Number of pages: 21


 {'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2004-12-13T15:22:54+00:00', 'author': 'DWest', 'moddate': '2014-10-24T15:09:14-06:00', 'title': 'R0501K_pdf.fm', 'source': 'documents/managing_oneself.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}

 all contents:  www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing On


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [26]:
import sys
from urllib import response
sys.path.append('../05_src/')

from utils.logger import get_logger
_logs = get_logger(__name__)

from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


prompt = f"""
    You are an AI assistant that summarizes books for AI professionals. 
    Given the following context from a book, do the following:
    
    1. Identify the book's author, and title.
    2. Determine it's Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. Determine how many stories are included in the book.
    4. Summary: a concise and succinct summary no longer than 1000 tokens. The summary should be written in a tone that is appropriate for an AI professional. The tone should be informative, engaging, and accessible, avoiding jargon and technical language whenever possible. The summary should highlight the key insights and takeaways from the book, and should be written in a way that is easy to understand and apply to the work of an AI professional.
   
    The book is the following: 
    <book>
    {all_text}
    </book>

    The fields of the object should be:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    Input Token Count: <input_token_count>
    Output Token Count: <output_token_count>
"""

system_prompt = "Output should be a Pydantic BaseModel object and change the tone to funny!"

user_prompt_response = client.responses.create(
    model = 'gpt-4o',
    input = prompt,
)

system_prompt_response = client.responses.create(
    model = 'gpt-4o',
    instructions=system_prompt,
    input = prompt,
)


from IPython.display import display, Markdown

print("User Prompt Response from the model:\n")
display(Markdown(user_prompt_response.output_text))

print("\n\n System Prompt Response from the model:\n")
display(Markdown(system_prompt_response.output_text))



User Prompt Response from the model:



**Title:** Managing Oneself  
**Author:** Peter F. Drucker

**Relevance:**  
"Managing Oneself" offers critical insights for AI professionals who seek to thrive in the fast-evolving knowledge economy. Understanding personal strengths, values, and performance preferences empowers AI experts to maximize their potential and effectively navigate career transitions. This self-awareness fosters innovation, collaboration, and long-term career satisfaction, crucial for personal growth and making meaningful contributions to the AI field.

**Summary:**  
Peter Drucker's "Managing Oneself" underscores the necessity for individuals to become the CEOs of their own careers. In today's knowledge economy, where opportunities abound, professionals must take charge of their personal and career development. Key to this is understanding one's strengths, learning styles, and values. Drucker advocates for feedback analysis to identify strengths and improve upon them rather than focusing on weaknesses. He outlines the importance of knowing whether one is a reader or listener and performing in contexts that suit one’s personal style. Drucker emphasizes how understanding oneself can lead to more meaningful work and significant contributions. He suggests asking critical questions about strengths, performance preferences, values, and finding suitable environments to thrive. This approach ensures not only excellence in work but also the satisfaction that comes from a well-aligned career path. Managing relationships and communication effectively within teams is also highlighted, with the recognition that trust and understanding are vital to organizational success. Drucker also addresses the second half of one's career, stressing that preparation and adaptation ensure continued growth and fulfillment.

**Tone:**  
Informative, engaging, and accessible.

**Input Token Count:**  
12402

**Output Token Count:**  
322



 System Prompt Response from the model:



```python
from pydantic import BaseModel

class BookSummary(BaseModel):
    title: str
    author: str
    relevance: str
    summary: str
    tone: str
    input_token_count: int
    output_token_count: int

book_summary = BookSummary(
    title="Managing Oneself",
    author="Peter F. Drucker",
    relevance="This book is essential for AI professionals as it teaches the fundamental skill of self-management in an era where career trajectories are no longer linear. By understanding one's strengths, values, and work style, AI professionals can maximize their contributions to projects and organizations.",
    summary=(
        "Peter Drucker's 'Managing Oneself' is about taking the reins of your own "
        "career in today's tumultuous work environment. AI professionals can learn to "
        "identify their strengths using techniques like feedback analysis and focus on "
        "what they excel at rather than average out weaknesses. The book emphasizes "
        "understanding how you work—whether you're a listener or reader, or prefer collaboration "
        "over solitary tasks. It stresses aligning one's career moves with personal values "
        "to avoid ending up as the proverbial square peg in a round hole. By transforming strengths "
        "into performance, AI pros like you can become true stars—not just another blinking cursor "
        "in the system. As you navigate a work life that spans decades, remember to consider and occasionally adjust "
        "your path, ensuring you stay engaged, productive, and sane, all while avoiding unnecessary cat memes."
    ),
    tone="Informative and engaging with a sprinkle of humor",
    input_token_count=10650,
    output_token_count=690
)

book_summary
```

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
